# Phase 6 — Untuned Boosting Model Training

This notebook trains CatBoost, LightGBM, and XGBoost on the fixed Phase 5 training rows and compares them on the fixed validation rows. Every model learns `log1p(selling_price)` and is scored after its predictions are converted back to Indian rupees.

The 2,287 test rows remain untouched. This is an untuned validation comparison—not final model selection.

## 1. Imports and project paths

In [ ]:
from pathlib import Path
import json
import sys

import catboost
import joblib
import lightgbm
import numpy as np
import pandas as pd
import xgboost
from catboost import CatBoostRegressor

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    CATBOOST_MODEL_PATH, LIGHTGBM_MODEL_PATH, MODEL_TRAINING_METADATA_PATH,
    MODEL_TRAINING_REPORT_PATH, MODEL_TRAINING_SUMMARY_PATH,
    PROCESSED_DATA_PATH, SPLIT_ASSIGNMENT_PATH, TABLES_DIR, XGBOOST_MODEL_PATH,
)
from src.features import BASE_FEATURE_COLUMNS, load_cleaned_features
from src.train_models import (
    COMPARISON_TABLE_FILENAME, PRICE_BAND_TABLE_FILENAME,
    _validated_predictions, load_fixed_split, prepare_catboost_features,
    run_model_training, split_training_and_validation,
)
from src.validate_data import file_sha256

print(f"Project root: {PROJECT_ROOT}")
print(f"Cleaned input: {PROCESSED_DATA_PATH}")
print(f"Fixed split: {SPLIT_ASSIGNMENT_PATH}")

## 2. Verify the fixed training design

The file hashes identify the exact cleaned data and split used. `split_training_and_validation` deliberately returns only training and validation data, so model code cannot accidentally receive the test rows.

In [ ]:
cleaned_hash_before = file_sha256(PROCESSED_DATA_PATH)
split_hash_before = file_sha256(SPLIT_ASSIGNMENT_PATH)
cars = load_cleaned_features(PROCESSED_DATA_PATH)
assignment = load_fixed_split(SPLIT_ASSIGNMENT_PATH, expected_rows=len(cars))
x_train, x_validation, y_train, y_validation, train_indices, validation_indices = (
    split_training_and_validation(cars, assignment)
)

display(assignment['split'].value_counts().reindex(['train', 'validation', 'test']).rename('rows').to_frame())
print(f"Training feature shape: {x_train.shape}")
print(f"Validation feature shape: {x_validation.shape}")
print(f"Test rows passed to model code: 0")
print(f"Cleaned SHA-256: {cleaned_hash_before}")
print(f"Split SHA-256:   {split_hash_before}")

## 3. Training approach

- **CatBoost** receives categorical columns explicitly. Category values become stable strings, while missing numeric values use medians learned only from training rows.
- **LightGBM and XGBoost** use training-only median/mode imputation plus `OneHotEncoder(handle_unknown="ignore")`. This lets future unseen category values pass safely.
- **All three** train on log price, use early stopping on validation log-RMSE, use random seed 42, and use no more than four CPU threads. Tree boosting does not need feature scaling.

Running the next cell retrains all three models and replaces the Phase 6 artifacts with a freshly validated set. On the tested machine it takes under one minute.

In [ ]:
phase6_summary = run_model_training()
print("Training complete.")
print(f"Preliminary validation winner: {phase6_summary['preliminary_best_untuned_model']}")
print(f"Test set evaluated: {phase6_summary['test_set']['evaluated']}")

## 4. Compare validation metrics in original rupees

MAE is the primary comparison metric because it has a direct business interpretation: the average absolute gap between a listing price and the estimated price. R² is useful context, but it is not the sole selection rule.

In [ ]:
comparison = pd.read_csv(TABLES_DIR / COMPARISON_TABLE_FILENAME)
comparison_display = comparison.copy()
for column in ['mae_inr', 'rmse_inr', 'median_absolute_error_inr']:
    comparison_display[column] = comparison_display[column].map(lambda value: f'₹{value:,.0f}')
comparison_display['r2'] = comparison_display['r2'].map(lambda value: f'{value:.4f}')
comparison_display['rmsle'] = comparison_display['rmsle'].map(lambda value: f'{value:.4f}')
comparison_display['training_seconds'] = comparison_display['training_seconds'].map(lambda value: f'{value:.1f}s')
comparison_display['mae_improvement_vs_dummy_percentage'] = comparison_display['mae_improvement_vs_dummy_percentage'].map(lambda value: f'{value:.1f}%')
display(comparison_display)

best_row = comparison.iloc[1:].sort_values('mae_inr').iloc[0]
print(f"{best_row['model']} has the lowest untuned validation MAE: ₹{best_row['mae_inr']:,.0f}.")
print(f"Its MAE is {best_row['mae_improvement_vs_dummy_percentage']:.1f}% below the dummy baseline.")

### Overall validation comparison

![Overall validation comparison](../reports/figures/14_model_validation_comparison.png)

## 5. Check performance by price band

Overall MAE can hide weak results on high-priced cars. The table and log-scale chart keep all four market segments visible. Phase 7 will add residual and category diagnostics.

In [ ]:
band_metrics = pd.read_csv(TABLES_DIR / PRICE_BAND_TABLE_FILENAME)
band_display = band_metrics.copy()
for column in ['mae_inr', 'rmse_inr', 'median_absolute_error_inr']:
    band_display[column] = band_display[column].map(lambda value: f'₹{value:,.0f}')
band_display['r2'] = band_display['r2'].map(lambda value: f'{value:.4f}')
band_display['rmsle'] = band_display['rmsle'].map(lambda value: f'{value:.4f}')
display(band_display)

### Validation MAE by price band

![Validation MAE by price band](../reports/figures/15_model_mae_by_price_band.png)

## 6. Reload the saved models and try an unseen category

A saved artifact is useful only if it can be reloaded and can handle future category values without crashing. This cell replaces brand and model with unseen examples, then predicts with every saved model.

In [ ]:
metadata = json.loads(MODEL_TRAINING_METADATA_PATH.read_text(encoding='utf-8'))
example = x_validation.head(1).copy()
example.loc[:, 'brand'] = 'Unknown Brand'
example.loc[:, 'model'] = 'Unknown Model'

catboost_model = CatBoostRegressor()
catboost_model.load_model(CATBOOST_MODEL_PATH)
catboost_input = prepare_catboost_features(
    example, metadata['catboost_numeric_fill_values']
)
reload_predictions = {
    'CatBoost': _validated_predictions(catboost_model.predict(catboost_input))[0]
}

for model_name, path in [('LightGBM', LIGHTGBM_MODEL_PATH), ('XGBoost', XGBOOST_MODEL_PATH)]:
    bundle = joblib.load(path)
    transformed = bundle['preprocessor'].transform(example)
    reload_predictions[model_name] = _validated_predictions(
        bundle['model'].predict(transformed)
    )[0]

display(pd.Series(reload_predictions, name='predicted_price').map(lambda value: f'₹{value:,.0f}').to_frame())
assert np.isfinite(list(reload_predictions.values())).all()
assert (np.asarray(list(reload_predictions.values())) >= 0).all()
print('All saved artifacts reloaded and handled unseen category strings.')

## 7. Final verification

These checks confirm that the inputs did not change, every advanced model beat the dummy benchmark, and no test evaluation occurred.

In [ ]:
assert file_sha256(PROCESSED_DATA_PATH) == cleaned_hash_before
assert file_sha256(SPLIT_ASSIGNMENT_PATH) == split_hash_before
assert comparison['model'].tolist() == ['Dummy baseline', 'CatBoost', 'LightGBM', 'XGBoost']
assert comparison.iloc[1:]['mae_inr'].lt(comparison.iloc[0]['mae_inr']).all()
assert phase6_summary['preliminary_best_untuned_model'] == 'CatBoost'
assert phase6_summary['training_contract']['hyperparameter_tuning_performed'] is False
assert phase6_summary['test_set']['evaluated'] is False
assert phase6_summary['test_set']['predictions_generated'] is False
assert phase6_summary['test_set']['metrics_computed'] is False
assert MODEL_TRAINING_REPORT_PATH.exists()
assert MODEL_TRAINING_SUMMARY_PATH.exists()

print('Phase 6 verification passed.')
print('Next: Phase 7 will inspect residuals, category errors, and the largest misses before tuning.')